In [ ]:
#pip install anthropic python-dotenv

In [ ]:
from dotenv import load_dotenv

"""
Reads a file named .env in your project folder and loads each KEY=value line into the environment variables of the running program. 
Your .env file should contain:
ANTHROPIC_API_KEY=your-key-here
"""
load_dotenv()

# Imports the Anthropic class from the official SDK (pip install anthropic) ....
from anthropic import Anthropic

"""
Creates the client object you use to call the API. You pass no key because the SDK automatically reads the ANTHROPIC_API_KEY environment 
variable, which is why load_dotenv() had to run before this line. The load_dotenv() call placed above the anthropic import isn't 
required for this to work, since the key is only read when Anthropic() runs. Putting all imports at the top is more conventional.
"""
client = Anthropic()

# FIX: Change from "claude-sonnet-5-0" to "claude-sonnet-5" or "claude-sonnet-5-5"
model = "claude-sonnet-5-5" 

"""
Calls the Messages API, the main endpoint for talking to Claude. It sends your request over the internet and waits for the full response. 
The result is stored in message.
"""
message = client.messages.create(
    # Tells the API which model to use. The name on the left is the parameter, and the model on the right is your variable.
    model=model,
    # A required cap on how long the reply can be. A token is a chunk of text, roughly three-quarters of an English word. 
    # If Claude reaches 1000 tokens, it stops mid-answer. It is a ceiling and not a target, so a short answer uses far fewer tokens.
    max_tokens=1000,

    
    #The conversation so far, as a list of message dictionaries. Each has a role ("user" for you, "assistant" for Claude) and content 
    #(the text). Here there is one user message. For a multi-turn chat, you append earlier messages to this list, because the API has no 
    #memory between calls. You send the whole history each time. The closing ) ends the create( call.
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

# message.content is a list of content blocks, not a plain string, because a reply can contain several blocks (text, tool calls and so on). 
# [0] takes the first block, and .text gets its string. For a simple text question there is one text block, so this prints 
# Claude's one-sentence answer.
print(message.content[0].text)


# This second message WONT return anything as Cluade doesn't have context to write another sentence.
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "Write another sentence"
        }
    ]
)
print(message.content[0])


In [ ]:
# Multi Conversation Flow ....

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )

    # Never index into content by position. Filter by type, and keep a fallback for types you don't recognize.
    for block in message.content:
        if block.type == "text":
            print("TEXT:", block.text)
        elif block.type == "thinking":
            print("THINKING (may be empty):", block.thinking)
        elif block.type == "tool_use":
            print("TOOL REQUEST:", block.name, block.input)
        else:
            print("OTHER BLOCK:", block.type)

    print(message.stop_reason, [b.type for b in message.content])   

    #return message.content[0].text
    # Collect only the text blocks; skip thinking blocks
    return "".join(block.text for block in message.content if block.type == "text")

    #         Part	                                   What it does
    # for block in message.content	    Walks through every block in the reply, in order
    # if block.type == "text"	        Keeps only blocks whose type is "text" and skips the rest
    # block.text	                    Reads the string from each kept block
    # "".join(...)	                    Glues the kept strings into one string (the "" means nothing is inserted between them)

In [ ]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)
print("Answer from Claude: ", answer)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)
print("Final Answer from Claude: ", final_answer)

In [ ]:
# Make an initial list of messages.
messages = []

# Use a 'while True' loop to run the chatbot forever
while True:
    # Get user input
    user_input = input("> ")
    print("> ", user_input)

    #Add user to the list of messages.
    add_user_message(messages, user_input)

    # Call Claude with 'chat' function
    answer = chat(messages)

    # Add generated text to the list of messages.
    add_assistant_message(messages, answer)

    # Print the generated text.
    print("--------\n")
    print(answer)
    print("\n--------")